# Prerequisites

- Technical
  - Python programming skills
  - Numpy skills
  - Basic text preprocessing skills: Understanding the purpose of lowercasing, tokenization, stop word removal, stemming...

- Theoretical
  - Term frequency (TF) & Document frequency (DF)
  - Inverse document frequency (IDF)
  - BM25 mechanisms: understanding the purpose of each term in the BM25's scoring family of functions: $IDF(q_i), TF(q_i, D), k_1, b, |D|, avgdl$

# Introduction

### 1. What is BM25?

**BM25** or **Best matching 25** algorithm is an information retrieval ranking algorithm used to determine how relevant is a document to a query. It is an improved version of the TF-IDF (Term frequency - Inverse document frequency) algorithm, and is the current industry-standard algorithm used by search engines and databases. It cares about exact and partial term matches, term frequency, and how rare a term is across documents.

### 2. What problem does BM25 solve?

Like any IR (information retrieval) algorithm the BM25 algorithm, the BM25 algorithm is a “bag of words” ranking function that ranks a collection of documents based on their relevance to a given user query. The only catch is that BM25 solves two massive problems found in earlier search methods (like standard TF-IDF or binary keyword matching):

- **The Diminishing Returns of Word Repetition (TF Saturation):** In a basic text search, if Document A mentions the word $car$ 2 times, and Document B mentions it 20 times, a naive algorithm assumes Document B is 10 times more relevant. BM25 recognizes that after a word appears a few times, successive appearances offer diminishing returns. It uses a saturation curve to ensure that frequent mentions of a word don't unfairly overpower other query terms.

- **The Short-vs-Long Document Bias (Length Normalization):** Long documents naturally contain more words, giving them a higher chance of matching query terms purely by accident. For example, a giant page might mention a word that belongs to $field_A$ in a footnote, while a tiny page is completely dedicated to that field. BM25 scales document scores based on how long a document is relative to the average document length in the dataset, penalizing long-winded documents and rewarding concise, highly relevant matches.

### 3. Real-World applications

- **Enterprise search engines:** the default scoring algorithm under the hood of many search engines is BM25, powering the search bars of platforms like Wikipedia, Github...
- **Hybrid RAG Pipelines:** In modern Generative AI and RAG systems, BM25 is heavily used alongside vector search.
- **Technical documentation search:** BM25 is highly effective for searching manuals or documentations where users hunt for explicit identifiers and symbols, for example function names, error codes command-line flags, etc. Which is the case we're going to be applying to day (Linux Manual search engine).

### 4. When vs When not to use BM25

| **Use BM25 when** | **Avoid BM25 when** |
| --- | --- |
| **Exact Matches Matter**: Excellent for searching technical terms, symbols, identifiers... | **Synonym heavy queries**: If a user searches for example 'automobile maintenance' BM25 will miss a document that has for example 'car repair' |
| **High efficiency**: It is very fast and memory efficient compared to deep-learning models. | **Conversational queries**: It struggles with long natural language questions where the semantic inten matters more. |
| **Cold Starts**: It requires zero training data or expensive GPU embeddings to be accurate. | **Cross-Lingual Search**: It can't map terms accross different languages without a translation layer |


# BM25 Algorithm

## 1. Mathematical Formulation

Before diving into the implementation of BM25 and to better understand it, it is necessary to understand the mathematical formula powering it:

Let a query $Q$ containing $q_1, \dots, q_n$ terms, the BM25 score of a document $D$ is:

$$score(Q, D) = \sum_{i=1}^{n} IDF(q_i) × \frac{f(q_i, D) \times (k_1 + 1)}{f(q_i, D) + k_1 \times (1 - b + b \times \frac{|D|}{avgdl})}$$

- $f(q_i, D)$ (Term Frequency): How many times the query token $q_i$ appears inside document $D$. Higher frequency means higher relevance, but scaled down by $k_1$.
- $k_1$ (TF Saturation Parameter): A tuning constant (usually between $1.2$ and $2.0$) that limits how much repeating a word can inflate a document's score. Once a term hits saturation, additional mentions yield diminishing returns.
- $b$ (Document Length Normalization Parameter): A constant (usually $0.75$) that controls how aggressively the algorithm penalizes long documents.
- $|D|$: the length of the current document in tokens.
- $\text{avgdl}$: the average token length across the entire corpus.
- If $\frac{|D|}{avgdl} > 1$, the document is longer than average and receives a penalty.
<br>

$IDF(q_i)$ is the weight of the query term $q_i$:

$$IDF(q_i) = ln[\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1]$$

- $N$: The total number of documents in the corpus.
- $n(q_i)$: The number of documents that contain the token $q_i$.
<br>
In the next sections we're going to explain two approaches to implement the BM25 algorithm, to observe the effects and benefits of production level optimizations.

## 2. Naive Approach

In this approach we're going to be calculating a $N × V$ ($N$: length of the corpus, $V$: number of terms in the vocabulary of the corpus), why this fails at scale:

- **Memory Complexity (O(N×V)):** Instantiates a big matrix tracking every vocabulary term across every document. In a document index containing 10,000 pages and a 50,000-word technical vocabulary, this matrix holds 500,000,000 elements—mostly filled with wasteful zeros.
- **Compute Bottleneck:** Iterating over every word in the global vocabulary during training to build the dense array, calculating combinations where tf=0 results in sluggish fit execution times that scale exponentially with large data.

## 3. Optimized Approach

In this approach we're going to introduce three improvements to our naive approach, to add more accuracy to the algorithm and cut down time and memory overhead:

- **Inverted Index:** instead of using the large matrix tracking millions of zeros, we transition to a dictionary mapping terms directly to matching document frequencies. At query time, the search engine scores the tiny slice of documents having the query terms.
- **Tokenization:** This is a tokenizer upgrade specific to our dataset (linux manual pages), since it contains technical system names (function_names, error codes, flags...) we're going to perform a token expantion where the technical terms are expanded into individual searchable keywords (eg. 'pthread_create' would be indexed as 'pthread_create', 'pthread' and 'create').
- **Title Boosting:** Words found in the title/filename of a document are indicate a high semantic match compared to words within the document's paragraphs. We're going to intercept the base BM25 scoring accumulator, to apply a weight multiplier (e.g., $3.0\times$) whenever a query term intersects with the document's title.

# Collecting man pages

In [ ]:
!unminimize

This system has been minimized by removing packages and content that are
not required on a system that users do not log into.

This script restores content and packages that are found on a default
Ubuntu server system in order to make this system more suitable for
interactive use.

Reinstallation of packages may fail due to changes to the system
configuration, the presence of third-party packages, or for other
reasons.

This operation may take some time.

Would you like to continue? [y/N] y



In [ ]:
!mkdir man-dump

In [ ]:
!for page in $(man -k . | awk '$2 ~ /^\(3/ {print $1}'); do man "$page" | col -bx > "man-dump/${page}.txt"; done

troff: <standard input>:337: warning [p 2, 0.3i]: cannot adjust line
  table wider than line width
troff: <standard input>:471: warning [p 3, 10.0i]: cannot adjust line
troff: <standard input>:61: warning [p 1, 6.0i, div '3tbd1,0', 0.0i]: can't break line
troff: <standard input>:303: warning [p 1, 5.5i]: cannot adjust line
troff: <standard input>:164: warning [p 3, 0.2i, div '3tbd1,0', 0.0i]: cannot adjust line
troff: <standard input>:166: warning [p 3, 0.2i, div '3tbd1,1', 0.0i]: cannot adjust line
troff: <standard input>:173: warning [p 3, 0.2i, div '3tbd2,0', 0.0i]: cannot adjust line
troff: <standard input>:175: warning [p 3, 0.2i, div '3tbd2,1', 0.0i]: cannot adjust line
troff: <standard input>:193: warning [p 3, 0.2i, div '3tbd4,1', 0.0i]: cannot adjust line
troff: <standard input>:202: warning [p 3, 0.2i, div '3tbd5,1', 0.0i]: cannot adjust line
troff: <standard input>:211: warning [p 3, 0.2i, div '3tbd6,1', 0.0i]: cannot adjust line
troff: <standard input>:218: warning [p 3, 0.

In [ ]:
def substr_between(s: str, start: str, end: str) -> tuple[int, int]:
  start_idx, end_idx = s.find(start), s.find(end)

  if start_idx != -1:
    start_idx = start_idx + len(start)
  else:
    start_idx = 0

  if end_idx != -1:
    end_idx = end_idx
  else:
    end_idx = len(s)

  return start_idx, end_idx

In [ ]:
import pandas as pd
import os

data = []

for page in os.listdir("man-dump/"):
  with open(f"man-dump/{page}") as f:
    file_content = f.read()
    start, end = substr_between(file_content, "NAME", "SEE ALSO")
    data.append([page.strip('.txt'), file_content[start: end]])

man_df = pd.DataFrame(data=data, columns=['title', 'content'])
man_df

,title,content
0,iswdigi,\n iswdigit - test for decimal digit wid...
1,cfgetospeed,"\n termios, tcgetattr, tcsetattr, tcse..."
2,ccos,"\n ccos, ccosf, ccosl - complex cosine f..."
3,numa_alloc_onnode,\n numa - NUMA policy library\n\nSYNOPSI...
4,idx_export_json,\n doctools::idx::export::json - JSON ex...
...,...,...
3805,ibv_inc_rkey,\n ibv_inc_rkey - creates a new rkey fro...
3806,libssh2_sftp_fsta,\n libssh2_sftp_fstat - convenience m...
3807,wprintf,"\n wprintf, fwprintf, swprintf, vwprintf..."
3808,semop,"\n semop, semtimedop - System V semaphor..."


In [ ]:
man_df[man_df['title'].str.startswith('pthread_create')]

,title,content
2918,pthread_create,\n pthread_create - create a new thread\...


# Naive Implementation

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import SnowballStemmer
import re
import numpy as np


class NaiveBM25:
  def __init__(self, k1=1.2, b=.3) -> None:
    self.k1 = 1.2
    self.b = .3

    self.vocab = []
    self._token_to_idx = {}
    self._idf = None
    self._score_mat = None
    self._stemmer = SnowballStemmer('english')

  def tokenize(self, text: str) -> list[str]:
    text = text.lower()

    token_pattern = r'-{1,2}[a-z0-9_-]+|[a-z0-9_-]+'
    raw_tokens = re.findall(token_pattern, text)
    return [
        self._stemmer.stem(token) for token in raw_tokens
        if token not in ENGLISH_STOP_WORDS
        and len(token) > 1
        ]

  def fit(self, docs: list[str]) -> None:
    n_docs = len(docs)
    unique_toks = set(tok for doc in docs for tok in self.tokenize(doc))
    self.vocab = sorted(list(unique_toks))
    self._token_to_idx = {token: i for i, token in enumerate(self.vocab)}

    tokenized_docs = [self.tokenize(doc) for doc in docs]

    doc_freq = np.zeros(len(self.vocab))
    freq_per_doc = np.zeros((n_docs, len(self.vocab)))
    doc_lens = np.zeros(n_docs)
    for i, tokenized_doc in enumerate(tokenized_docs):
      doc_lens[i] = len(tokenized_doc)
      for term in set(tokenized_doc):
        doc_freq[self._token_to_idx[term]] += 1
        freq_per_doc[i][self._token_to_idx[term]] += tokenized_doc.count(term)

    avgdl = np.mean(doc_lens)

    self._idf = np.zeros(len(self.vocab))
    for term in self.vocab:
      term_idx = self._token_to_idx[term]
      self._idf[term_idx] = np.log((n_docs - doc_freq[term_idx] + .5) / (doc_freq[term_idx] + .5) + 1)

    self._score_mat = np.zeros((n_docs, len(self.vocab)))
    for i, tokenized_doc in enumerate(tokenized_docs):
      for term in set(tokenized_doc):
        term_idx = self._token_to_idx[term]

        numerator = freq_per_doc[i][term_idx] * (self.k1 + 1)
        denominator = freq_per_doc[i][term_idx] + self.k1 * (1 - self.b + self.b * (doc_lens[i] / avgdl))
        self._score_mat[i][term_idx] = self._idf[term_idx] * (numerator / denominator)

  def top_n_docs(self, query: str, top_n: int=10) -> list[int]:
    if not query:
      return []

    q_terms = [self._token_to_idx[term] for term in self.tokenize(query) if term in self.vocab]
    scores = np.sum(self._score_mat[:, q_terms], axis=1)
    return np.argsort(scores)[::-1][:top_n].tolist()

# Optimized Implementation

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter, defaultdict
from nltk.stem import SnowballStemmer
import numpy as np
import math
import re

class OptimizedBM25:
  def __init__(self, k1: float = 1.6, b: float = 0.3) -> None:
    self.k1 = k1
    self.b = b
    self._stemmer = SnowballStemmer('english')

    self._n_docs = 0
    self._avgdl = 0.0
    self._doc_lens = []
    self._idf = {}

    self._docs_titles = {}
    self._inverted_index = defaultdict(dict)

  def tokenize(self, text: str) -> list[str]:
    text = text.lower()

    token_pattern = r'-{1,2}[a-z0-9_-]+|[a-z0-9_-]+'
    raw_tokens = re.findall(token_pattern, text)
    expanded_tokens = []

    for tok in raw_tokens:
      if tok in ENGLISH_STOP_WORDS:
        continue

      expanded_tokens.append(tok)
      if '-' in tok or '_' in tok:
        split = re.split('[_|-]+', tok)
        expanded_tokens.extend([s for s in split if s not in ENGLISH_STOP_WORDS and len(s) > 1])

    return [
        self._stemmer.stem(token) for token in expanded_tokens
        if len(token) > 1
        ]

  def fit(self, docs: list[str], docs_titles: list[str]=[]) -> None:
    self._n_docs = len(docs)
    self._doc_lens = []
    doc_frequencies = defaultdict(int)

    for i, doc in enumerate(docs):
      tokenized_doc = self.tokenize(doc)
      self._doc_lens.append(len(tokenized_doc))

      term_counts = Counter(tokenized_doc)

      for term, freq in term_counts.items():
        self._inverted_index[term][i] = freq
        doc_frequencies[term] += 1

    if docs_titles and len(docs_titles) == self._doc_lens:
      self._docs_titles = {i: self.tokenize(doc_title) for i, doc_title in enumerate(docs_titles)}

    self._avgdl = np.mean(self._doc_lens) if self._n_docs > 0 else 0

    for term, df in doc_frequencies.items():
      self._idf[term] = math.log(((self._n_docs - df + 0.5) / (df + 0.5)) + 1.0)

  def top_n_docs(self, query: str, top_n: int = 10) -> list[int]:
    if not query or self._n_docs == 0:
      return []

    query_terms = self.tokenize(query)
    scores = np.zeros(self._n_docs)

    for term in query_terms:
      if term not in self._inverted_index:
        continue

      for i, tf in self._inverted_index[term].items():
        title_boost = 1.0
        numerator = tf * (self.k1 + 1)
        denominator = tf + self.k1 * (1 - self.b + self.b * (self._doc_lens[i] / self._avgdl))

        if self._docs_titles and term in self._docs_titles[i]:
          title_boost = 3.0

        scores[i] += self._idf[term] * (numerator / denominator)
        scores[i] *= title_boost

    return np.argsort(scores)[::-1][:top_n].tolist()

In [ ]:
docs = man_df['content'].to_list()

model = NaiveBM25()
model.fit(docs)

In [ ]:
titles = man_df['title'].to_list()
docs = man_df['content'].to_list()

model1 = OptimizedBM25()
model1.fit(docs, titles)

In [ ]:
query = "how to create a thread?"

In [ ]:
top_docs = model.top_n_docs(query)

top_docs

[2886, 2918, 1300, 990, 3661, 421, 1521, 3452, 3658, 3298]

In [ ]:
man_df.iloc[top_docs]['title'].tolist()

['pthread_getattr_np',
 'pthread_create',
 'pthread_attr_setdetachstate',
 'pthread_attr_getdetachstate',
 'pthread_attr_setsigmask_np',
 'pthread_attr_getsigmask_np',
 'zmq_ctx_se',
 'pthread_attr_getinheritsched',
 'pthread_attr_setinheritsched',
 'pthread_getschedparam']

In [ ]:
top_docs = model1.top_n_docs(query)

top_docs

[2918, 275, 2886, 990, 1300, 3604, 421, 3661, 3584, 3298]

In [ ]:
man_df.iloc[top_docs]['title'].tolist()

['pthread_create',
 'pthread_key_create',
 'pthread_getattr_np',
 'pthread_attr_getdetachstate',
 'pthread_attr_setdetachstate',
 'imer_create',
 'pthread_attr_getsigmask_np',
 'pthread_attr_setsigmask_np',
 'pthread_setschedparam',
 'pthread_getschedparam']